# 5. Memory
    - 메모리를 추가해야 이어지는 질문도 기억할 수 있다.
    - Open AI API는 stateless. 메모리가 없어 아무런 내용도 저장하지 않는다.
    - 여러 경로의 DB와 memory 간 intergrations 가능.

## 5.0 ConversationBufferMemory
    - 전체 대화 내용을 저장한다.
    - Text 자동완성 시 사용
    - 대화 내용이 길어질수록 메모리도 같이 커지기 때문에 비효율적이다.
    - 모델에 대화 기록을 전부 보내기 때문에 비용이 많이 든다.

## 5.1 ConversationBufferWindowMemory
    - 최근 메시지만 저장한다는 것이 단점.
    - 메모리를 특정 크기로 유지할 수 있다는 것이 장점.

## 5.2 ConversationSummaryMemory
    - message를 요약하여 저장한다.
    - llm을 사용하기 때문에 비용이 든다.
    - 초반에는 저장공간을 더 차지하지만, 대화가 많아질수록 토큰 양이 줄어든다.

## 5.3 ConversationSummaryBufferMemory
    - ConversationSummaryMemory + ConversationBufferMemory
    - 메시지 수를 저장하다가 limit에 다다른 순간 오래된 메시지를 summarize 한다.

## 5.4 Conversation Knowledge Graph Memory
    - llm을 사용하여 중요한 내용(entity)만 요약한다.

## 5.4 Conversation Token Buffer Memory
    - ConversationBufferWindowMemory와 비슷한데, interaction 최대 값 대신 token 총 양을 지정하여 오래된 메모리는 버린다.

## 5.4 Entity Memory
    - save history and entities.

In [ ]:
# 5.0 ConversationBufferMemory

from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory() # this returns text.
memory = ConversationBufferMemory(return_messages=True) # this returns chat message class.
memory.save_context({"input":"Hi!"},{"output":"How are you"})
memory.load_memory_variables({})

In [ ]:
# 5.1 ConversationBufferWindowMemory

from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=1, # 저장할 메시지 수
)

def add_message(input, output):
    memory.save_context({"input":input}, {"output":output})

add_message(1,1)
add_message(2,2)

memory.load_memory_variables({})

In [ ]:
# 5.2 ConversationSummaryMemory

from langchain.memory import ConversationSummaryMemory
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(temperature=0.1)
memory = ConversationSummaryMemory(llm=llm)

def add_message(input, output):
    memory.save_context({"input":input}, {"output":output})

def get_history():
    return memory.load_memory_variables({})

add_message("Moscare, Mom", "Hey, Macarena")
add_message("Dibigo Dibigo", "Make a wish")

get_history()

In [ ]:
# 5.3 ConversationSummaryBufferMemory

from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=150, # 메시지 요약 전 최대 토큰 한계.
    return_messages=True,
)

def add_message(input, output):
    memory.save_context({"input":{input}}, {"output":output})

def get_history():
    memory.load_memory_variables({})

add_message("Hi it's too hot in here.", "Here as well.")
get_history()

In [3]:
# 5.4 ConversationKGMemory

from langchain.chat_models.openai import ChatOpenAI
from langchain.memory import ConversationKGMemory

llm = ChatOpenAI(temperature=0.1)
memory = ConversationKGMemory(
    llm=llm,
    return_messages=True,
)

def add_message(input, output):
    memory.save_context({"input":{input}}, {"output":{output}})

add_message("Nicolas lives in Spain.", "He likes Kimchi too much to live in Spain.")

memory.load_memory_variables({"input": "What Nicolas likes?"})

{'history': [SystemMessage(content='On Nicolas: Nicolas lives in Spain.')]}

## 5.5 Memory on LLMChain VS 5.7 LCEL Based Memory
### 5.5 LLMChain
    - LLMChain: off-the-shelf(general purpose) chain
    - 자동으로 LLM으로부터 response를 가져와서 memory를 update하지만, 그 메모리를 prompt에 넣는 건 수동.
1. Memory on PromptTemplate
2. #5.6 Chat Based Memory

In [ ]:
# 1. Memory on PromptTemplate

from langchain.chat_models.openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import LLMChain 

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history" # template 안에 메모리 내용을 넣는다.
)

template = """
    You are a helpful AI talking to human.

    {chat_history}
    Human: {question}
    You:
"""

# 일반적인 체인
chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=PromptTemplate.from_template(template),
    verbose=True, # Chain의 Prompt Log 확인 가능
)

chain.predict(question="Seoul is in rainy today.")
chain.predict(question="How's the weather today in Seoul?")

memory.load_memory_variables({})



> Entering new LLMChain chain...
Prompt after formatting:
Seoul is in rainy today.

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
How's the weather today in Seoul?

> Finished chain.


"I'm sorry, I am an AI assistant and I do not have real-time information. I recommend checking a reliable weather website or app for the most up-to-date weather information in Seoul."

In [ ]:
# 2. Chat Based Memory

from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import LLMChain

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    ("system", "You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="chat_history"), # 불특정 다수의 메시지를 담을 수 있다.
    ("human", "{question}"),
)

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=True,
)

### 5.7 LCEL Based Memory
    - Custom Chain에 LCEL을 사용해서 메모리 추가
    - Add the history to the prompt manually.

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import  ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

def load_memory(_): # ignore the input, question in the chain.invoke
    return memory.load_memory_variables({})["history"]

        # 2                                             | # 3
chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

# 1
def invoke_chain(question):
    result = chain.invoke({"question":question})
    memory.save_context({"input":question}, {"output":result.content})
    print(result)